In [ ]:
from ugot import ugot
import cv2
import numpy as np
import time
from IPython.display import display, clear_output, Image

got = ugot.UGOT()
got.initialize("192.168.1.193")

got.load_models(
    [
        "color_recognition",  # detects dominant colors
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode",  # AprilTag recognition
    ]
)

In [ ]:
def face_text_preview():
    # Grab the latest camera frame as raw bytes
    frame = got.read_camera_data()

    # Decode the bytes into an OpenCV image array
    nparr = np.frombuffer(frame, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Overlay any detected text in the top-left corner
    name = got.get_words_result()
    if name:
        cv2.putText(img, f"text: {name}", (20, 40), 0, 0.8, (0, 255, 0), 2)

    # Overlay face recognition data if any faces are detected.
    # Each face entry is: [name, center_x, center_y, height]
    faces = got.get_face_recognition_total_info()
    if faces:
        face = faces[0]  # Only look at the first detected face
        cv2.putText(img, f"face: {face}", (20, 70), 0, 0.8, (0, 255, 0), 2)

    # Encode the annotated frame as JPEG and display it inline in the notebook
    _, jpeg = cv2.imencode(".jpg", img)
    clear_output(wait=True)
    display(Image(data=jpeg.tobytes()))

In [ ]:
# Ensure the correct models are loaded and the camera is open
# !!!This will NOT be needed on the actual competition. It is here for testing purposes only.
got.open_camera()

try:
    while True:
        face_text_preview()

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()
    clear_output(wait=True)

In [ ]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

In [ ]:
def pick_up():
    got.mechanical_clamp_release()
    got.mechanical_joint_control(0, 0, -65, 500)
    time.sleep(2)
    got.mechanical_clamp_close()
    time.sleep(1)
    got.mechanical_joint_control(0, 0, 0, 500)
    time.sleep(1)
    got.mechanical_single_joint_control(1, 45, 1500)
    time.sleep(1)

def put_down():
    got.mechanical_joint_control(0, 0, -75, 1500)
    time.sleep(2)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(0, 0, 0, 500)

In [ ]:
pick_up()

In [ ]:
put_down()

In [ ]:
TARGET_NAME = "Ryan"

face_find_and_approach(
    gap=15,
    target_name=TARGET_NAME,
    turn_spd=15,
    strafe_spd=10,
    fwd_spd=10,
    height=120,
    adjust_turn=5,
)

got.mecanum_move_speed_times(1, 10, 5, 1)

put_down()